# Preview: Auto Tagging Support Tickets Using LLM


This notebook uses a Large Language Model (LLM) to automatically classify support tickets by predicting the top three most relevant categories using Zero Shot and Few Shot prompting.


## Import Libraries and Set Up LLM
This cell loads the necessary libraries and initializes our language model text-generation environment.

In [2]:
import pandas as pd
import numpy as np
from transformers import pipeline

# Initialize an open, highly capable instruction-tuned model
classifier_llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device=0)

print("LLM environment successfully initialized!")

LLM environment successfully initialized!


## Load and Inspect the Support Tickets Dataset
This cell creates a mock dataset representing free-text customer support tickets to simulate a real-world company helpdesk database.

In [13]:
# Creating a sample dataset matching diverse customer complaints
data = {
    'ticket_id': [101, 102, 103, 104, 105, 106, 107],
    'Text': [
        "Hey, I was charged twice for my monthly subscription. Please refund the second transaction.",
        "My screen goes completely black whenever I try to open the desktop app. Please help.",
        "I forgot my password and the reset link email is not arriving in my inbox.",
        "The application is lagging terribly when importing CSV files, it keeps freezing up my system.",
        "I need to update my billing card but the settings page keeps loading infinitely.",
        "Your system locked my account after 3 failed login attempts. Unban me ASAP.",
        "The application crashed my laptop completely when exporting the analytics data report."
    ]
}

df = pd.DataFrame(data)
df.head()

,ticket_id,Text
0,101,"Hey, I was charged twice for my monthly subscr..."
1,102,My screen goes completely black whenever I try...
2,103,I forgot my password and the reset link email ...
3,104,The application is lagging terribly when impor...
4,105,I need to update my billing card but the setti...


## Implement Zero-Shot Prompting Strategy
This cell defines a strict prompt instructing the LLM to categorize the ticket instantly without providing any prior examples.

In [14]:
def get_zero_shot_prompt(ticket_text):
    return f"""You are an expert customer support agent system. Analyze the following support ticket and categorize it.
Allowed Categories: Billing, Technical Support, Account Security, Performance.

Ticket: "{ticket_text}"

Output exactly the top 3 most probable categories ranked from highest to lowest probability.
Format your output strictly like this: 1. Category, 2. Category, 3. Category.
CRITICAL: Do not add any introductory text, explanations, or trailing notes. Stop immediately after listing the categories.

Top 3 Tags:"""

# Test the prompt structure on the first ticket
print(get_zero_shot_prompt(df['Text'][0]))

You are an expert customer support agent system. Analyze the following support ticket and categorize it.
Allowed Categories: Billing, Technical Support, Account Security, Performance.

Ticket: "Hey, I was charged twice for my monthly subscription. Please refund the second transaction."

Output exactly the top 3 most probable categories ranked from highest to lowest probability. 
Format your output strictly like this: 1. Category, 2. Category, 3. Category. 
CRITICAL: Do not add any introductory text, explanations, or trailing notes. Stop immediately after listing the categories.

Top 3 Tags:


## Implement Few-Shot Prompting Strategy
This cell upgrades our strategy by adding a few strict example pairs inside the prompt to guide the LLM's behavioral pattern.

In [23]:
def get_few_shot_prompt(ticket_text):
    return f"""You are an expert customer support agent system. Analyze the following support ticket and categorize it.
Allowed Categories: Billing, Technical Support, Account Security, Performance.

Example 1:
Ticket: "I cannot log into my profile and I think my account was hacked."
Top 3 Tags: 1. Account Security, 2. Technical Support, 3. Performance

Example 2:
Ticket: "I need an invoice copy for my last payment change."
Top 3 Tags: 1. Billing, 2. Account Security, 3. Technical Support

Now process this new ticket:
Ticket: "{ticket_text}"

Output exactly the top 3 categories ranked from highest to lowest probability.
Format your output strictly like this: 1. Category, 2. Category, 3. Category
CRITICAL: Do not write anything else. Stop typing immediately after the 3rd category.

Top 3 Tags:"""

# Test the prompt structure on the first ticket
print(get_few_shot_prompt(df['Text'][0]))

You are an expert customer support agent system. Analyze the following support ticket and categorize it.
Allowed Categories: Billing, Technical Support, Account Security, Performance.

Example 1:
Ticket: "I cannot log into my profile and I think my account was hacked."
Top 3 Tags: 1. Account Security, 2. Technical Support, 3. Performance

Example 2:
Ticket: "I need an invoice copy for my last payment change."
Top 3 Tags: 1. Billing, 2. Account Security, 3. Technical Support

Now process this new ticket:
Ticket: "Hey, I was charged twice for my monthly subscription. Please refund the second transaction."

Output exactly the top 3 categories ranked from highest to lowest probability. 
Format your output strictly like this: 1. Category, 2. Category, 3. Category
CRITICAL: Do not write anything else. Stop typing immediately after the 3rd category.

Top 3 Tags:


## Execute LLM Inference Evaluation
This cell feeds our tickets through both the Zero-Shot and Few-Shot setups to compare the model's parsing behavior.

In [35]:
# Extract the tokenizer directly from your existing pipeline object
pipeline_tokenizer = classifier_llm.tokenizer

zero_shot_results = []
few_shot_results = []

print("Running sanitized inference with tokenizer pass... Please wait.")

for text in df['Text']:
    # 1. Zero-Shot
    zs_prompt = get_zero_shot_prompt(text)
    zs_out = classifier_llm(
        zs_prompt,
        max_new_tokens=40,
        return_full_text=False,
        tokenizer=pipeline_tokenizer,
        stop_strings=["\n", "Explanation:", "This ticket"]
    )[0]['generated_text']
    zero_shot_results.append(zs_out.strip())

    # 2. Few-Shot
    fs_prompt = get_few_shot_prompt(text)
    fs_out = classifier_llm(
        fs_prompt,
        max_new_tokens=40,
        return_full_text=False,
        tokenizer=pipeline_tokenizer,
        stop_strings=["\n", "Explanation:", "This ticket"]
    )[0]['generated_text']
    few_shot_results.append(fs_out.strip())

df['ZeroShot_Tags'] = zero_shot_results
df['FewShot_Tags'] = few_shot_results

print("Inference completed successfully!")

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running sanitized inference with tokenizer pass... Please wait.


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Inference completed successfully!


## Compare Performance Results
This cell outputs the final data table side-by-side so we can observe how adding few-shot examples cleaned up the classification.

In [36]:
# Review the final comparative classifications
df[['Text', 'ZeroShot_Tags', 'FewShot_Tags']]

,Text,ZeroShot_Tags,FewShot_Tags
0,"Hey, I was charged twice for my monthly subscr...","Billing, Technical Support, Performance","1. Billing, 2. Technical Support, 3. Account S..."
1,My screen goes completely black whenever I try...,"Technical Support, Account Security, Billing","1. Performance, 2. Technical Support, 3. Accou..."
2,I forgot my password and the reset link email ...,"Account Security, Technical Support, Billing","1. Account Security, 2. Technical Support, 3. ..."
3,The application is lagging terribly when impor...,"Technical Support, Performance, Account Security","1. Performance, 2. Technical Support, 3. Billing"
4,I need to update my billing card but the setti...,"Technical Support, Billing, Account Security","1. Technical Support, 2. Billing, 3. Account S..."
5,Your system locked my account after 3 failed l...,"Performance, Technical Support, Account Security","1. Account Security, 2. Technical Support, 3. ..."
6,The application crashed my laptop completely w...,"1. Application Issues, 2. System Crash, 3. Dat...","1. Technical Support, 2. Performance, 3. Billing"


## Export Classification Results to CSV
This cell converts our final comparative evaluation dataframe into a physical, downloadable CSV file for production logs or external auditing.

In [37]:
# Save the final dataframe results into a clean, physical CSV file
csv_filename = "tagged_support_tickets_evaluation.csv"
df.to_csv(csv_filename, index=False)

print(f"File successfully created and saved as '{csv_filename}'!")

File successfully created and saved as 'tagged_support_tickets_evaluation.csv'!


### Project Summary

* Engineered an automated support ticket classification system using the lightweight open-source `Qwen2.5-1.5B-Instruct` model to process unstructured customer complaints into structured, ranked business tags.
* Python, Hugging Face Transformers, Hugging Face Accelerate, Pandas, and an active NVIDIA T4 GPU runtime.
* Evaluated the operational efficiency of Zero-Shot Inference versus Few-Shot Prompt Engineering across a 7-ticket sample suite covering complex, overlapping technical issues.
* Identified an inherent behavioral tendency where small instruction-tuned models append extensive textual explanations. This issue was completely mitigated by implementing real-time token termination via native tokenizer `stop_strings=["\n"]` injection.
* **Key Analytical Findings:**
    * *Zero-Shot Flaws:* Suffered from schema non-compliance, failed to output requested numbering templates, and completely hallucinated invalid category labels outside the restricted list boundaries (e.g., Ticket 107).
    * *Few-Shot Success:* Achieved 100% adherence to the structural schema standard (`1. Category, 2. Category, 3. Category`), strictly respected category constraints, and demonstrated superior priority ranking accuracy (e.g., properly identifying account lockouts as an immediate Account Security risk).
* Produced a complete, machine-readable pipeline output file automated into a flat-file log structure named `tagged_support_tickets_evaluation.csv` for backend system consumption.

In [45]:
import nbformat

file_name = "SupportTicket_TaggingPipeline.ipynb"

with open(file_name, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Delete the master widget metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Loop through cells to delete ONLY loading bars while keeping text prints
for cell in nb.cells:
    if 'outputs' in cell:
        clean_outputs = []
        for output in cell['outputs']:
            # Check if the output is a hidden Jupyter widget/loading bar
            is_widget = False
            if 'data' in output:
                if 'application/vnd.jupyter.widget-view+json' in output['data']:
                    is_widget = True

            # If it is a normal text print, keep it
            if not is_widget:
                clean_outputs.append(output)

        cell['outputs'] = clean_outputs

# Save the updated notebook
with open(file_name, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print("Success! The loading bars are gone, but your print outputs are 100% safe.")

FileNotFoundError: [Errno 2] No such file or directory: 'SupportTicket_TaggingPipeline.ipynb'

In [46]:
import requests
import json

# This code snippet fetches the current notebook's path in Google Colab
# and extracts its filename.

# Get the Jupyter server info (contains the notebook's path)
try:
    response = requests.get('http://localhost:8888/api/sessions')
    sessions = json.loads(response.text)
    current_notebook_path = None
    for session in sessions:
        if 'notebook' in session and session['notebook']['path'].endswith('.ipynb'):
            current_notebook_path = session['notebook']['path']
            break

    if current_notebook_path:
        # Extract the filename from the path
        current_notebook_filename = current_notebook_path.split('/')[-1]
        print(f"The exact filename of this notebook is: {current_notebook_filename}")
    else:
        print("Could not determine the current notebook's filename. Please check if you are running in a Colab environment.")
except requests.exceptions.ConnectionError:
    print("Could not connect to the Jupyter server. This usually happens if you're not running in a Colab/Jupyter environment or if there's a temporary issue.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Could not connect to the Jupyter server. This usually happens if you're not running in a Colab/Jupyter environment or if there's a temporary issue.


In [47]:
# This cell lists all files in the current directory.
# Look for the .ipynb file that corresponds to this notebook.
!ls -F

# Once you see the exact filename, copy it and paste it into the 'file_name' variable in cell '-CdH_NpuzGNi'.

sample_data/  tagged_support_tickets_evaluation.csv
